# 🚀 Lanzar una app FastAPI + React desde Google Drive — Colab + Cloudflare

**Plantilla genérica · v3** · configurada para: *ExportBot 2.0 (v2.0.0b2) · exportaciones de Colombia · GIC ProColombia · A08: monorepos*

Toma la **carpeta del proyecto que usted descomprimió en su Google Drive**, la copia
al disco local del runtime, prepara el entorno, lanza el backend con Uvicorn y lo
expone con una **URL pública temporal** vía
[TryCloudflare](https://developers.cloudflare.com/cloudflare-one/connections/connect-networks/do-more-with-tunnels/trycloudflare/)
(sin cuenta, sin token, sin instalar nada en su computador). Las llaves LLM viven en
los **Secrets de Colab (🔑)**: el visitante del enlace nunca ve ni aporta una clave.

**Qué cambia frente a la versión anterior (diagnóstico A07.1):**

1. 🧪 **Causa raíz de la "lentitud eterna" corregida.** El `package-lock.json` del
   release traía sus 87 URLs `resolved` apuntando a un espejo privado
   (`packages.applied-caas-gateway1.internal.api.openai.org`) inalcanzable desde
   Colab: `npm ci` reintentaba paquete por paquete, en silencio, durante minutos, y
   luego caía a un `npm install` completo. Ahora `sanear_lockfiles()` reescribe esos
   espejos al registro oficial **antes** de cualquier `npm`, y se detiene con detalle
   si encuentra un host que no sabe reparar.
2. 🎁 **Frontend precompilado.** El ZIP del release ahora incluye `dist/` ya
   compilado. Si `dist/index.html` llega desde Drive, se **omiten Node, npm y el
   build completos** (los 3 pasos más lentos). Lanzar pasa de ~8–10 min a ~1–3 min.
3. 📟 **Latidos y tiempos.** Toda tarea larga (pip, npm, build) escribe a un log e
   imprime avance cada 10 s con el tiempo transcurrido: nunca más minutos de
   silencio sin saber si avanza o se colgó. Todo tiene **timeout** explícito.
4. ⬇️ **Node por tarball oficial.** Si hay que instalar Node (solo sin `dist/`), se
   usa el tarball de nodejs.org (~30 s) en lugar de NodeSource + apt (~2–4 min),
   con NodeSource como respaldo automático.
5. ✅ **Verificación extremo a extremo.** Tras abrir el túnel, el notebook consulta
   la URL **pública** hasta recibir HTTP 200 de `/api/v1/salud`: la celda termina
   diciendo explícitamente si el lanzamiento quedó operativo o no.

**Estructura:** 🅰️ *Celda A* = configuración (lo único que edita) · 🅱️ *Celda B* =
pipeline genérico (no se toca) · Pasos 1→3 · utilidades de operación al final.

> ⚠️ **Límites no negociables — léalos antes de usar**
> 1. **Términos de Colab.** La [FAQ de Colab](https://research.google.com/colaboratory/faq.html)
>    prohíbe usar los runtimes para servir aplicaciones web de forma sostenida. Una
>    demo corta, con usted presente y el notebook como actividad principal, es el
>    único uso defendible — y aun así Google puede terminar la sesión **sin aviso**.
>    Para algo permanente use Railway con el `Dockerfile` del proyecto (ver el final).
> 2. **La URL es pública mientras viva.** Cualquiera con el enlace consulta el
>    inventario que viaja en el proyecto. No lance aquí datos que no puedan salir
>    del área. Si define `APP_TOKEN` en los Secrets, la API exigirá ese token.
> 3. **Sesión efímera.** Al cerrar Colab (o al expirar el túnel) el enlace muere.
>    Eso es una característica, no un defecto.

## 🅰️ Celda A · Configuración — lo único que se edita

Para reutilizar esta plantilla en **otro proyecto** solo cambie estas constantes
(ruta en Drive, carpeta del backend, módulo ASGI, ruta de salud y variables de
entorno). El resto del notebook es genérico.

In [1]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELDA A — CONFIGURACIÓN (único lugar que se edita)                   ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── 1. De dónde sale el proyecto ─────────────────────────────────────
#     "drive_carpeta": usted descomprimió el ZIP y subió la CARPETA a Drive (recomendado).
#     "drive_zip"    : el ZIP está en Drive y el notebook lo descomprime.
FUENTE_PROYECTO = "drive_carpeta"
RUTA_DRIVE      = "/content/drive/MyDrive/ProColombia/exportbot"
RUTA_ZIP_DRIVE  = "/content/drive/MyDrive/ProColombia/exportbot_v2.0.0b2.zip"  # solo si FUENTE_PROYECTO="drive_zip"

# ── 2. Anatomía de la app (genérico para cualquier FastAPI + frontend) ──
BACKEND_DIR     = "backend"            # carpeta con el ASGI (imports planos → cwd aquí)
APP_ASGI        = "main:app"           # módulo:variable para Uvicorn
RUTA_SALUD      = "/api/salud"         # endpoint que confirma el arranque
PUERTO          = 8000
COMPILAR_FRONTEND = True               # solo actúa si NO llegó dist/ precompilado
NODE_INSTALAR   = "22.12.0"            # versión exacta del tarball oficial si toca instalar
NODE_FALLBACK   = "22"                 # mínimo si package.json no declara "engines"

# ── 3. Entorno de la app ─────────────────────────────────────────────
#     Variables fijas que el backend leerá. Los SECRETOS van en Secrets (🔑).
ENV_APP = {
    "ARRANQUE_ESTRICTO": "false",       # demo: la app sube y declara qué falta en /api/salud
    "PROVEEDOR_REDACCION": "cortex",    # la redacción por defecto no sale de Snowflake
}
#     Secretos de Colab (🔑) que, SI EXISTEN, se pasan al backend con el mismo nombre.
#     Mínimo para operar: SF_ACCOUNT, SF_USER, (SF_PAT o SF_PRIVATE_KEY_B64_1),
#     SF_ROLE, SF_WAREHOUSE y SF_SEMANTIC_VIEW (o SF_SEMANTIC_MODEL_FILE).
SECRETS_PASSTHROUGH = [
    "SF_ACCOUNT", "SF_USER", "SF_PAT",
    "SF_PRIVATE_KEY_B64_1", "SF_PRIVATE_KEY_PASSPHRASE_1",
    "SF_PRIVATE_KEY_B64_2", "SF_PRIVATE_KEY_PASSPHRASE_2",
    "SF_ROLE", "SF_WAREHOUSE", "SF_DATABASE", "SF_SCHEMA", "SF_HOST",
    "SF_SEMANTIC_VIEW", "SF_SEMANTIC_MODEL_FILE",
    "SF_ESQUEMA_TELEMETRIA", "SF_CORTEX_MODELO",
    "ADMIN_TOKEN", "APP_TOKEN",
    "OPENAI_API_KEY", "GEMINI_API_KEY", "GROQ_API_KEY",
    "OPENROUTER_API_KEY", "ANTHROPIC_API_KEY",
]

# ── 4. Saneo de lockfiles (hallazgo A07.1) ───────────────────────────
#     True: reescribe URLs de espejos privados/file:// a los índices públicos
#     oficiales ANTES de instalar nada (solo en la copia local; su Drive no se toca).
REPARAR_LOCKFILES = True

# ── 5. Rutas del runtime (no suele hacer falta tocarlas) ─────────────
DIR_TRABAJO = "/content/app"        # copia local del proyecto (Drive es lento para I/O)
DIR_LOGS    = "/content/logs"

print("✅ Configuración cargada ·", FUENTE_PROYECTO, "· puerto", PUERTO,
      "· salud", RUTA_SALUD,
      "· saneo de lockfiles", "ON" if REPARAR_LOCKFILES else "OFF")


✅ Configuración cargada · drive_carpeta · puerto 8000 · salud /api/salud · saneo de lockfiles ON


## 🅱️ Celda B · Pipeline genérico — no modificar

Funciones de todo el ciclo: montar Drive → traer el proyecto → **sanear lockfiles** →
preparar Python (y Node solo si falta `dist/`) → lanzar Uvicorn + túnel → **verificar
la URL pública** → operar (estado, logs, relanzar, detener). Sin efectos al
ejecutarla: solo define herramientas.

In [2]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELDA B — PIPELINE GENÉRICO (no modificar)                           ║
# ╚══════════════════════════════════════════════════════════════════════╝
from __future__ import annotations

import json as _json
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

PROCESOS: dict[str, subprocess.Popen] = {}
#: OJO: ``dist`` YA NO se ignora — el release lo trae precompilado y con él se
#: omiten Node, npm y el build (los tres pasos más lentos del lanzamiento).
IGNORAR_COPIA = {".git", "node_modules", "__pycache__", ".venv", "venv",
                 ".pytest_cache", ".mypy_cache", ".ruff_cache", ".ipynb_checkpoints",
                 "resultados", "*.egg-info", ".coverage", "coverage.xml"}

# ── Tiempos y límites (sin números mágicos en la lógica) ─────────────
TIMEOUT_SALUD_S = 120        # espera máxima a que el backend responda /salud
TIMEOUT_TUNEL_S = 90         # espera máxima a que cloudflared publique la URL
TIMEOUT_PUBLICA_S = 45       # espera máxima a que la URL pública responda 200
TIMEOUT_PARADA_S = 8         # gracia antes de forzar el cierre de un proceso
TIMEOUT_CMD_S = 300          # tope por defecto de cualquier comando corto
TIMEOUT_PIP_S = 900          # tope de la instalación Python
TIMEOUT_NPM_S = 900          # tope de npm ci / npm install
TIMEOUT_BUILD_S = 600        # tope del build de Vite
HTTP_TIMEOUT_S = 60          # descargas puntuales (nodejs.org, cloudflared)
NODE_MAX_MEM_MB = 4096       # memoria para Node durante el build de Vite
INTERVALO_SONDEO_S = 2       # pausa entre sondeos de salud/túnel
LATIDO_S = 10                # cada cuánto imprime avance una tarea larga
LINEAS_LOG_DEFECTO = 40      # líneas de log a mostrar en diagnóstico
REGISTRO_NPM_OFICIAL = "https://registry.npmjs.org/"

#: Cola de ruta de un tarball npm: ``pkg/-/pkg-1.2.3.tgz`` o ``@scope/pkg/-/…``.
#: Los espejos tipo Artifactory conservan esta cola tal cual, así que basta con
#: re-anclar el prefijo del host al registro oficial.
_RE_TGZ_NPM = re.compile(r'https://[^"\s]+?/((?:@[^"/\s]+/)?[^"/\s]+/-/[^"/\s]+\.tgz)')
#: Línea ``paquete @ file:///…/paquete-1.2.3-….whl`` de un lock de pip.
_RE_WHEEL_FILE = re.compile(
    r"^\s*([A-Za-z0-9][A-Za-z0-9._-]*)\s*@\s*file://\S*?/"
    r"([A-Za-z0-9_.]+)-([0-9][^-\s]*)-[^\s]*\.whl\S*\s*$"
)


def _cron(inicio: float) -> str:
    """Formatea el tiempo transcurrido desde ``inicio`` como ``'37s'``."""
    return f"{time.time() - inicio:.0f}s"


def _run(cmd, cwd=None, env=None, check: bool = True,
         timeout_s: int = TIMEOUT_CMD_S) -> subprocess.CompletedProcess:
    """Ejecuta un comando corto capturando SIEMPRE la salida.

    Lección A07.1: sin captura, en Colab el stderr de un subproceso se pierde y
    un fallo queda mudo (fue exactamente lo que ocultó la causa raíz anterior).
    Si el comando falla, aquí se imprime su cola y se levanta ``RuntimeError``.

    Args:
        cmd: Lista de argumentos del comando.
        cwd: Directorio de trabajo.
        env: Entorno del subproceso (hereda el actual si es ``None``).
        check: Si ``True``, un código ≠ 0 levanta ``RuntimeError``.
        timeout_s: Tope duro; evita celdas colgadas para siempre.

    Returns:
        El ``CompletedProcess`` con ``stdout``/``stderr`` capturados.

    Raises:
        RuntimeError: Si ``check`` y el comando falla o agota el tiempo.
    """
    print("  $", " ".join(map(str, cmd)))
    try:
        r = subprocess.run(list(map(str, cmd)), cwd=cwd, env=env, text=True,
                           capture_output=True, timeout=timeout_s)
    except subprocess.TimeoutExpired as error:
        raise RuntimeError(
            f"Tiempo agotado ({timeout_s}s): {' '.join(map(str, cmd))}"
        ) from error
    if check and r.returncode != 0:
        print(((r.stdout or "") + (r.stderr or ""))[-2000:])
        raise RuntimeError(f"Comando falló ({r.returncode}): {' '.join(map(str, cmd))}")
    return r


def _cola_log(log: Path, n: int = 15) -> None:
    """Imprime las últimas ``n`` líneas de ``log`` (si existe)."""
    if log.exists():
        print("\n".join(log.read_text(errors="ignore").splitlines()[-n:]))


def _run_latido(cmd, log: Path, cwd=None, env=None,
                timeout_s: int = TIMEOUT_NPM_S, latido_s: int = LATIDO_S) -> int:
    """Ejecuta una tarea LARGA con latido: log en disco + avance cada ``latido_s``.

    Nunca más minutos de silencio: cada latido imprime el tiempo transcurrido y
    la última línea del log, para distinguir "avanza" de "se colgó". Devuelve el
    código de salida (no levanta): el llamador decide si hay plan B.

    Args:
        cmd: Comando a ejecutar.
        log: Archivo donde se vuelca stdout+stderr completos.
        cwd: Directorio de trabajo.
        env: Entorno del subproceso.
        timeout_s: Tope duro; al agotarse se mata el proceso (código 124).
        latido_s: Intervalo entre impresiones de avance.

    Returns:
        Código de salida del proceso (124 si se agotó el tiempo).
    """
    print("  $", " ".join(map(str, cmd)), f"→ log: {log.name}")
    inicio = time.time()
    with log.open("w") as fh:
        p = subprocess.Popen(list(map(str, cmd)), cwd=cwd, env=env,
                             stdout=fh, stderr=subprocess.STDOUT, text=True)
        ultimo_latido = inicio
        while p.poll() is None:
            time.sleep(1)
            if time.time() - inicio > timeout_s:
                p.kill()
                print(f"⛔ Tiempo agotado ({timeout_s}s). Últimas líneas:")
                _cola_log(log)
                return 124
            if time.time() - ultimo_latido >= latido_s:
                ultimo_latido = time.time()
                lineas = log.read_text(errors="ignore").splitlines()
                ultima = lineas[-1][:88] if lineas else "(sin salida aún)"
                print(f"    … {_cron(inicio)} · {ultima}")
    if p.returncode != 0:
        print(f"⚠️ Terminó con código {p.returncode}. Últimas líneas:")
        _cola_log(log)
    return p.returncode


def montar_drive() -> None:
    """Monta Google Drive en ``/content/drive`` (silencioso fuera de Colab)."""
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
        print("✅ Drive montado.")
    except ModuleNotFoundError:
        print("ℹ️ Fuera de Colab: RUTA_DRIVE se usa como ruta local.")


def _secreto(nombre: str) -> str | None:
    """Lee un secreto de Colab (🔑); fuera de Colab, de las variables de entorno."""
    try:
        from google.colab import userdata  # type: ignore
        return userdata.get(nombre) or None
    except Exception:
        return os.environ.get(nombre) or None


# ── Saneo de lockfiles (hallazgo A07.1) ──────────────────────────────
def sanear_lockfiles(proyecto: Path) -> None:
    """Repara lockfiles envenenados por espejos privados, ANTES de instalar nada.

    Causa raíz de la "lentitud eterna" (A07.1): el ``package-lock.json`` del
    release traía 87 URLs ``resolved`` de un Artifactory interno inalcanzable
    desde Colab; ``npm ci`` reintentaba en silencio durante minutos y luego caía
    a un ``npm install`` completo. Además ``requirements.lock.txt`` fijaba un
    paquete a un wheel ``file:///tmp/…`` inexistente. Esto ocurre cuando el lock
    se regenera en un entorno cuyo npm/pip pasa por un proxy corporativo.

    Solo toca la COPIA LOCAL del runtime (nunca su Drive). Es idempotente. Si
    tras reparar queda algún host que no es el registro oficial, se DETIENE con
    el detalle (fail-fast) en lugar de dejar que npm agonice en silencio.

    Args:
        proyecto: Raíz local del proyecto (la copia en ``DIR_TRABAJO``).

    Raises:
        RuntimeError: Si ``REPARAR_LOCKFILES`` es ``False`` y hay veneno, o si
            queda alguna referencia que el saneo no sabe reescribir.
    """
    problemas: list[str] = []

    lock_npm = proyecto / "package-lock.json"
    if lock_npm.exists():
        texto = lock_npm.read_text(encoding="utf-8")
        nuevo = _RE_TGZ_NPM.sub(lambda m: REGISTRO_NPM_OFICIAL + m.group(1), texto)
        cambios = sum(1 for a, b in zip(texto.splitlines(), nuevo.splitlines()) if a != b)
        if cambios:
            if not REPARAR_LOCKFILES:
                problemas.append(f"package-lock.json: {cambios} URLs de espejo privado")
            else:
                lock_npm.write_text(nuevo, encoding="utf-8")
                _json.loads(lock_npm.read_text(encoding="utf-8"))  # sigue siendo JSON válido
                print(f"🧹 package-lock.json: {cambios} URLs re-ancladas a "
                      f"{REGISTRO_NPM_OFICIAL} (integridad sha512 intacta).")
        restantes = [u for u in re.findall(r'"resolved":\s*"(https://[^"]+)"',
                                           lock_npm.read_text(encoding="utf-8"))
                     if not u.startswith(REGISTRO_NPM_OFICIAL)]
        if restantes:
            problemas.append(f"package-lock.json: hosts desconocidos {sorted(set(restantes))[:3]}")

    lock_pip = proyecto / "requirements.lock.txt"
    if lock_pip.exists():
        lineas, reparadas = [], 0
        for linea in lock_pip.read_text(encoding="utf-8").splitlines():
            m = _RE_WHEEL_FILE.match(linea)
            if m:
                nombre, version = m.group(2).replace("_", "-"), m.group(3)
                if REPARAR_LOCKFILES:
                    lineas.append(f"{nombre}=={version}")
                    reparadas += 1
                    continue
                problemas.append(f"requirements.lock.txt: '{m.group(1)}' fijado a file://")
            elif "file://" in linea:
                problemas.append(f"requirements.lock.txt: referencia file:// no reconocida: {linea[:60]}")
            lineas.append(linea)
        if reparadas:
            lock_pip.write_text("\n".join(lineas) + "\n", encoding="utf-8")
            print(f"🧹 requirements.lock.txt: {reparadas} referencia(s) file:// → PyPI.")

    if problemas:
        detalle = "\n   · ".join(problemas)
        raise RuntimeError(
            "Lockfiles con referencias no instalables desde Colab:\n   · " + detalle +
            "\nActive REPARAR_LOCKFILES=True en la Celda A o corrija el release."
        )
    if not REPARAR_LOCKFILES:
        print("↷ REPARAR_LOCKFILES=False: lockfiles sin veneno detectado, se continúa.")


def localizar_proyecto(raiz: Path) -> Path:
    """Encuentra la carpeta del proyecto (v3: tolera monorepos).

    Marcadores: ``pyproject.toml`` + ``{BACKEND_DIR}/main.py`` +
    ``package.json`` en la raíz **o** en ``frontend/`` (diagnóstico A08).
    """
    raiz = Path(raiz)
    candidatos, pistas = [], []
    for pyproject in raiz.rglob("pyproject.toml"):
        carpeta = pyproject.parent
        if any(parte in IGNORAR_COPIA for parte in carpeta.parts):
            continue
        falta = []
        if not ((carpeta / "package.json").exists()
                or (carpeta / "frontend" / "package.json").exists()):
            falta.append("package.json (en la raíz o en frontend/)")
        if not (carpeta / BACKEND_DIR / "main.py").exists():
            falta.append(f"{BACKEND_DIR}/main.py")
        if falta:
            pistas.append(f"{carpeta} → falta: {', '.join(falta)}")
        else:
            candidatos.append(carpeta)
    if len(candidatos) != 1:
        hijos = ([p.name for p in sorted(raiz.iterdir()) if p.is_dir()][:8]
                 if raiz.exists() else ["(la ruta no existe)"])
        raise RuntimeError(
            f"Se esperaba UN proyecto bajo {raiz} y se encontraron "
            f"{len(candidatos)}: {[str(c) for c in candidatos]}\n"
            f"  · Subcarpetas de primer nivel vistas: {hijos}\n"
            f"  · pyproject.toml sin marcadores completos: {pistas[:5] or 'ninguno'}\n"
            f"  · Requisito: pyproject.toml + {BACKEND_DIR}/main.py + package.json "
            "(raíz o frontend/). Si acaba de subir a Drive, espere a que termine "
            "de sincronizar y reintente."
        )
    return candidatos[0]


def traer_proyecto() -> Path:
    """Copia el proyecto de Drive (carpeta o zip) al disco local y lo sanea.

    Returns:
        Raíz local verificada del proyecto (bajo ``DIR_TRABAJO``).

    Raises:
        FileNotFoundError: Si no existe la ruta/ZIP configurados en la Celda A.
        RuntimeError: Si falta un archivo esencial o el saneo encuentra veneno
            que no puede reparar.
    """
    inicio = time.time()
    montar_drive()
    destino = Path(DIR_TRABAJO)
    if destino.exists():
        shutil.rmtree(destino)
    destino.mkdir(parents=True)

    if FUENTE_PROYECTO == "drive_carpeta":
        origen = Path(RUTA_DRIVE)
        if not origen.exists():
            raise FileNotFoundError(
                f"No existe {origen}. Descomprima el ZIP del proyecto y suba la "
                "carpeta a esa ruta de Drive, o ajuste RUTA_DRIVE en la Celda A."
            )
        base = localizar_proyecto(origen)
        print(f"📁 Copiando {base} → {destino} (sin {sorted(IGNORAR_COPIA)[:4]}…)")
        shutil.copytree(base, destino / base.name, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns(*IGNORAR_COPIA))
    elif FUENTE_PROYECTO == "drive_zip":
        zip_path = Path(RUTA_ZIP_DRIVE)
        if not zip_path.exists():
            candidatos = sorted(Path("/content/drive/MyDrive").glob("*gestion_conocimiento*.zip"))
            if len(candidatos) == 1:
                zip_path = candidatos[0]
                print(f"ℹ️ Usando el único ZIP encontrado: {zip_path.name}")
            else:
                raise FileNotFoundError(f"No se encontró {RUTA_ZIP_DRIVE} (candidatos: "
                                        f"{[c.name for c in candidatos]})")
        print(f"📦 Descomprimiendo {zip_path.name} → {destino}")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extractall(destino)
    else:
        raise ValueError(f"FUENTE_PROYECTO desconocida: {FUENTE_PROYECTO!r}")

    proyecto = localizar_proyecto(destino)
    n = sum(1 for _ in proyecto.rglob("*") if _.is_file())
    for esencial in ("pyproject.toml", "package.json", f"{BACKEND_DIR}/main.py"):
        if not (proyecto / esencial).exists():
            raise RuntimeError(f"Falta {esencial} en el proyecto.")
    sanear_lockfiles(proyecto)
    _avisar_faltantes_manifiesto(proyecto)
    dist_ok = ((proyecto / "dist" / "index.html").exists() or (proyecto / "frontend" / "dist" / "index.html").exists())
    print(f"✅ Proyecto listo en {proyecto} · {n} archivos · {_cron(inicio)}")
    print("🎁 Frontend precompilado:",
          "✅ dist/ presente → se OMITEN Node, npm y build" if dist_ok
          else "— no viene; se compilará en el Paso 2")
    return proyecto


def _avisar_faltantes_manifiesto(proyecto: Path) -> None:
    """Aviso temprano (no bloqueo): ¿la copia de Drive perdió archivos del release?

    RELEASE_MANIFEST.json viaja dentro del árbol y lista cada archivo del
    release. Las subidas de carpetas a Drive a veces sueltan archivos (caso real
    2026-07-18: faltó 1 baseline y el publicador reventó minutos después con un
    pytest críptico). La demo puede correr igual, así que aquí solo se AVISA con
    la lista exacta; el publicador sí bloquea.
    """
    ruta = proyecto / "RELEASE_MANIFEST.json"
    if not ruta.exists():
        return
    try:
        esperados = set(_json.loads(ruta.read_text(encoding="utf-8")).get("files_sha256") or {})
    except (ValueError, OSError):
        return
    presentes = {str(p.relative_to(proyecto)) for p in proyecto.rglob("*") if p.is_file()}
    faltan = sorted(esperados - presentes)
    if faltan:
        muestra = ", ".join(faltan[:4]) + (" …" if len(faltan) > 4 else "")
        print(f"⚠️ OJO: a la copia le faltan {len(faltan)} archivo(s) del release "
              f"(¿subida a Drive incompleta?): {muestra}")
        print("   La demo puede funcionar, pero PUBLICAR así se bloqueará. "
              "Remedio: reemplace la carpeta de Drive con el ZIP del release.")
    else:
        print(f"🪪 Integridad del release: {len(esperados)}/{len(esperados)} archivos presentes.")


# ── Python: instalación acelerada con uv (respaldo: pip) ─────────────
def _uv_disponible(py: str) -> str | None:
    """Devuelve la ruta de ``uv`` (instalándolo si hace falta) o ``None``."""
    ruta = shutil.which("uv")
    if ruta:
        return ruta
    r = subprocess.run([py, "-m", "pip", "install", "-q", "uv"],
                       capture_output=True, text=True, timeout=180)
    return shutil.which("uv") if r.returncode == 0 else None


def preparar_python(proyecto: Path) -> str:
    """Instala el paquete y el runtime del servidor en el Python del runtime.

    En Colab NO se usa ``python3 -m venv``: su arranque de pip (ensurepip) falla
    y deja un venv SIN pip — la causa histórica del 'No module named uvicorn'.
    El runtime de Colab ya es efímero y aislado, así que se instala directo en
    él, con ``uv`` para acelerar 5–10× (si uv no está disponible cae a pip sin
    cambiar el resultado). La instalación editable (``-e``) valida el empaquetado y deja los
    módulos reales de ``backend/`` enlazados al código fuente del proyecto.

    Args:
        proyecto: Raíz local del proyecto.

    Returns:
        Ruta del intérprete Python que ejecutará el backend.

    Raises:
        RuntimeError: Si al final la distribución o los módulos reales no importan.
    """
    inicio = time.time()
    py = sys.executable
    piezas: list[str] = ["-e", str(proyecto)]
    req_backend = proyecto / BACKEND_DIR / "requirements.txt"
    if req_backend.exists():
        piezas += ["-r", str(req_backend)]

    uv = _uv_disponible(py)
    if uv:
        print("📦 Instalando entorno Python con uv (rápido)…")
        r = _run([uv, "pip", "install", "-q", "--python", py, *piezas],
                 check=False, timeout_s=TIMEOUT_PIP_S)
        if r.returncode != 0:
            print("⚠️ uv falló; reintento con pip clásico…")
            uv = None
    if not uv:
        print("📦 Instalando entorno Python con pip…")
        _run([py, "-m", "pip", "install", "-q", *piezas], timeout_s=TIMEOUT_PIP_S)

    # Fail-fast: distribución y módulos reales deben importar con ESTE Python.
    codigo_chequeo = (
        "from importlib.metadata import version; "
        "assert version('exportbot'); "
        "import uvicorn, fastapi, config, main, orquestador, schemas"
    )
    chequeo = subprocess.run([py, "-c", codigo_chequeo],
                             capture_output=True, text=True, cwd=proyecto)
    if chequeo.returncode != 0:
        print(chequeo.stdout, chequeo.stderr)
        raise RuntimeError(
            "La instalación no dejó importables ExportBot, FastAPI o sus módulos reales "
            "(ver salida arriba)."
        )
    print(f"✅ Entorno Python listo ({py}) · {_cron(inicio)}")
    return py


# ── Node: solo si NO llegó dist/ precompilado ────────────────────────
def node_requerido(proyecto: Path) -> tuple[int, ...]:
    """Versión MÍNIMA de Node como tupla, según ``package.json → engines``.

    Devuelve p. ej. ``(22, 12)`` para ``">=22.12"`` (no solo el mayor), de modo
    que la comparación no acepte por error un Node 22.5 cuando se exige 22.12.
    """
    try:
        engines = _json.loads((proyecto / "package.json").read_text())["engines"]["node"]
        numeros = re.findall(r"\d+", engines)
    except Exception:
        numeros = []
    if not numeros:
        numeros = re.findall(r"\d+", NODE_FALLBACK)
    return tuple(int(n) for n in numeros[:3]) or (22,)


def _version_node_actual() -> tuple[int, ...]:
    """Versión de Node instalada como tupla, o ``()`` si no hay Node."""
    if not shutil.which("node"):
        return ()
    salida = subprocess.run(["node", "--version"], capture_output=True, text=True).stdout
    return tuple(int(x) for x in re.findall(r"\d+", salida)[:3])


def _instalar_node_tarball(version: str) -> bool:
    """Instala Node desde el tarball oficial de nodejs.org (~30 s, x86_64).

    Mucho más rápido que NodeSource (evita ``apt-get update``). Devuelve ``True``
    si tras extraer, ``node --version`` alcanza la versión pedida.
    """
    url = f"https://nodejs.org/dist/v{version}/node-v{version}-linux-x64.tar.xz"
    destino = Path("/tmp/node.tar.xz")
    try:
        print(f"⬇️ Node v{version} (tarball oficial nodejs.org)…")
        urllib.request.urlretrieve(url, destino)
        _run(["tar", "-xJf", str(destino), "-C", "/usr/local", "--strip-components=1"],
             timeout_s=180)
        return _version_node_actual() >= tuple(int(x) for x in version.split("."))
    except Exception as error:
        print(f"⚠️ Tarball no disponible ({error}); respaldo: NodeSource (más lento)…")
        return False


def asegurar_node(proyecto: Path) -> None:
    """Garantiza un Node que cumpla el mínimo de ``engines`` del proyecto.

    Primero el tarball oficial (rápido); si falla, NodeSource + apt (respaldo).

    Raises:
        RuntimeError: Si tras ambos intentos Node no alcanza el mínimo.
    """
    minimo = node_requerido(proyecto)
    actual = _version_node_actual()
    if actual >= minimo:
        print(f"✅ Node {'.'.join(map(str, actual)) or '—'} cumple "
              f"(requiere ≥{'.'.join(map(str, minimo))}).")
        return
    if not _instalar_node_tarball(NODE_INSTALAR):
        script = urllib.request.urlopen(
            f"https://deb.nodesource.com/setup_{minimo[0]}.x",
            timeout=HTTP_TIMEOUT_S).read()
        setup = Path("/tmp/node_setup.sh")
        setup.write_bytes(script)
        _run(["bash", str(setup)], timeout_s=TIMEOUT_CMD_S)
        _run(["apt-get", "install", "-y", "nodejs"], timeout_s=TIMEOUT_CMD_S)
    nueva = _version_node_actual()
    if nueva < minimo:
        raise RuntimeError(
            f"Node instalado {'.'.join(map(str, nueva)) or '—'} no alcanza el mínimo "
            f"{'.'.join(map(str, minimo))}."
        )
    print(f"✅ Node {'.'.join(map(str, nueva))} instalado.")


def compilar_frontend(proyecto: Path) -> None:
    """Compila el SPA con Vite SOLO si no llegó ``dist/`` precompilado.

    Con latido y timeout en npm y en el build; ``npm ci`` cae a ``npm install``
    si el lock quedó desincronizado. Los logs completos quedan en ``DIR_LOGS``.

    Raises:
        RuntimeError: Si el build termina sin ``dist/index.html``.
    """
    if not COMPILAR_FRONTEND:
        print("↷ COMPILAR_FRONTEND=False: se omite el build.")
        return
    if ((proyecto / "dist" / "index.html").exists() or (proyecto / "frontend" / "dist" / "index.html").exists()):
        print("♻️ dist/index.html presente: se omiten Node, npm y build "
              "(borre dist/ para forzar la recompilación).")
        return
    inicio = time.time()
    asegurar_node(proyecto)
    Path(DIR_LOGS).mkdir(parents=True, exist_ok=True)
    entorno = dict(os.environ, NODE_OPTIONS=f"--max-old-space-size={NODE_MAX_MEM_MB}")
    log_npm = Path(DIR_LOGS) / "npm.log"
    if (proyecto / "package-lock.json").exists():
        rc = _run_latido(["npm", "ci", "--no-audit", "--no-fund"], log_npm,
                         cwd=proyecto, timeout_s=TIMEOUT_NPM_S)
        if rc != 0:
            print("⚠️ 'npm ci' falló (lock desincronizado o red); reintento con 'npm install'…")
            rc = _run_latido(["npm", "install", "--no-audit", "--no-fund"], log_npm,
                             cwd=proyecto, timeout_s=TIMEOUT_NPM_S)
            if rc != 0:
                raise RuntimeError("npm no pudo instalar dependencias (ver log arriba).")
    else:
        rc = _run_latido(["npm", "install", "--no-audit", "--no-fund"], log_npm,
                         cwd=proyecto, timeout_s=TIMEOUT_NPM_S)
        if rc != 0:
            raise RuntimeError("npm install falló (ver log arriba).")
    rc = _run_latido(["npm", "run", "build"], Path(DIR_LOGS) / "build.log",
                     cwd=proyecto, env=entorno, timeout_s=TIMEOUT_BUILD_S)
    if rc != 0 or not ((proyecto / "dist" / "index.html").exists() or (proyecto / "frontend" / "dist" / "index.html").exists()):
        raise RuntimeError(
            "El build del frontend falló (ver log arriba). En Colab suele ser memoria "
            "o red intermitente: reintente esta celda."
        )
    print(f"✅ Frontend compilado (dist/) · {_cron(inicio)}")


def preparar_entorno(proyecto: Path) -> str:
    """Python primero, frontend después. Devuelve el intérprete a usar."""
    py = preparar_python(proyecto)
    compilar_frontend(proyecto)
    return py


# ── Túnel y lanzamiento ──────────────────────────────────────────────
def asegurar_cloudflared() -> Path:
    """Descarga el binario de ``cloudflared`` si no está presente y lo hace ejecutable."""
    binario = Path("/usr/local/bin/cloudflared")
    if binario.exists():
        return binario
    print("⬇️ Descargando cloudflared…")
    url = ("https://github.com/cloudflare/cloudflared/releases/latest/"
           "download/cloudflared-linux-amd64")
    urllib.request.urlretrieve(url, binario)
    binario.chmod(0o755)
    return binario


def _env_app(proyecto: Path) -> dict:
    """Entorno del backend: variables fijas de ``ENV_APP`` + secretos presentes.

    Los secretos de ``SECRETS_PASSTHROUGH`` se propagan solo si existen; se
    imprimen sus NOMBRES (nunca sus valores).
    """
    env = dict(os.environ)
    env.update({k: str(v) for k, v in ENV_APP.items()})
    presentes = []
    for nombre in SECRETS_PASSTHROUGH:
        valor = _secreto(nombre)
        if valor:
            env[nombre] = valor
            presentes.append(nombre)
    print("🔑 Secretos presentes:", presentes or "ninguno (modo sin proveedor servidor)")
    return env


def esperar_salud(puerto: int, ruta: str, timeout_s: int = TIMEOUT_SALUD_S) -> dict:
    """Sondea ``ruta`` local hasta HTTP 200 o hasta ``timeout_s``.

    Raises:
        TimeoutError: Si el backend no responde 200 dentro de ``timeout_s``.
    """
    inicio = time.time()
    while time.time() - inicio < timeout_s:
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{puerto}{ruta}", timeout=3) as r:
                if r.status == 200:
                    return _json.loads(r.read().decode())
        except Exception:
            pass
        time.sleep(INTERVALO_SONDEO_S)
    logs("backend", n=25)
    raise TimeoutError(f"El backend no respondió {ruta} en {timeout_s}s (ver logs arriba).")


def _url_tunel(log: Path, timeout_s: int = TIMEOUT_TUNEL_S) -> str:
    """Extrae la URL pública ``*.trycloudflare.com`` del log de cloudflared.

    Raises:
        TimeoutError: Si cloudflared no publica una URL dentro de ``timeout_s``.
    """
    patron = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
    inicio = time.time()
    while time.time() - inicio < timeout_s:
        if log.exists():
            m = patron.search(log.read_text(errors="ignore"))
            if m:
                return m.group(0)
        time.sleep(INTERVALO_SONDEO_S)
    logs("tunel", n=25)
    raise TimeoutError("cloudflared no publicó URL (ver log arriba).")


def _verificar_publica(url: str, ruta: str, timeout_s: int = TIMEOUT_PUBLICA_S) -> bool:
    """Confirma extremo a extremo que la URL PÚBLICA responde 200 en ``ruta``."""
    inicio = time.time()
    while time.time() - inicio < timeout_s:
        try:
            with urllib.request.urlopen(url + ruta, timeout=5) as r:
                if r.status == 200:
                    return True
        except Exception:
            pass
        time.sleep(INTERVALO_SONDEO_S)
    return False


def lanzar(proyecto: Path) -> str:
    """Backend + túnel + verificación pública. Devuelve la URL pública.

    La celda termina con un veredicto explícito: URL verificada (HTTP 200 desde
    afuera) o advertencia con el diagnóstico a mirar. Ese era el vacío anterior:
    imprimía la URL sin comprobar que el lanzamiento hubiera quedado operativo.
    """
    inicio = time.time()
    detener(silencioso=True)
    Path(DIR_LOGS).mkdir(parents=True, exist_ok=True)
    py = sys.executable
    entorno = _env_app(proyecto)
    # El paquete quedó editable; PYTHONPATH a la raíz por si el subproceso no lo hereda.
    entorno["PYTHONPATH"] = str(proyecto) + os.pathsep + entorno.get("PYTHONPATH", "")
    log_b = Path(DIR_LOGS) / "backend.log"
    with log_b.open("w") as fh:
        PROCESOS["backend"] = subprocess.Popen(
            [py, "-m", "uvicorn", APP_ASGI, "--host", "0.0.0.0",
             "--port", str(PUERTO)],
            cwd=proyecto / BACKEND_DIR, env=entorno,
            stdout=fh, stderr=subprocess.STDOUT,
        )
    salud = esperar_salud(PUERTO, RUTA_SALUD)
    resumen = {k: salud.get(k) for k in ("version", "total_artefactos", "origen_datos")
               if k in salud}
    print(f"✅ Backend arriba en :{PUERTO} · {resumen or salud}")

    cf = asegurar_cloudflared()
    log_t = Path(DIR_LOGS) / "tunel.log"
    with log_t.open("w") as fh:
        PROCESOS["tunel"] = subprocess.Popen(
            [str(cf), "tunnel", "--url", f"http://127.0.0.1:{PUERTO}", "--no-autoupdate"],
            stdout=fh, stderr=subprocess.STDOUT,
        )
    url = _url_tunel(log_t)
    print(f"🔎 Verificando la URL pública ({RUTA_SALUD})…")
    ok = _verificar_publica(url, RUTA_SALUD)
    print("\n" + "═" * 66)
    print(f"🌐 URL PÚBLICA:  {url}")
    print("═" * 66)
    if ok:
        print(f"✅ VERIFICADO extremo a extremo: HTTP 200 en {RUTA_SALUD} · "
              f"total {_cron(inicio)}")
    else:
        print("⚠️ El túnel existe pero la URL pública aún no responde 200. "
              "Espere unos segundos y pruebe el enlace; si persiste: "
              "logs('tunel') y logs('backend').")
    if _secreto("APP_TOKEN"):
        print("🔐 APP_TOKEN activo: las llamadas a /api exigen el header X-App-Token.")
    print("Compártala solo mientras dure la demo. Deténgala con detener().")
    return url


def estado() -> None:
    """Imprime si el backend y el túnel siguen vivos en esta sesión."""
    for nombre, p in PROCESOS.items():
        print(f"· {nombre}: {'vivo' if p.poll() is None else f'terminado ({p.returncode})'}")
    if not PROCESOS:
        print("· sin procesos lanzados en esta sesión")


def logs(cual: str = "backend", n: int = LINEAS_LOG_DEFECTO) -> None:
    """Muestra las últimas ``n`` líneas del log de ``backend``/``tunel``/``npm``/``build``."""
    ruta = Path(DIR_LOGS) / f"{cual}.log"
    if not ruta.exists():
        print(f"(no hay {ruta})")
        return
    print(f"── últimas {n} líneas de {ruta.name} " + "─" * 20)
    print("\n".join(ruta.read_text(errors="ignore").splitlines()[-n:]))


def detener(silencioso: bool = False) -> None:
    """Detiene backend y túnel (mata el enlace público). Idempotente."""
    for nombre, p in list(PROCESOS.items()):
        if p.poll() is None:
            p.terminate()
            try:
                p.wait(timeout=TIMEOUT_PARADA_S)
            except subprocess.TimeoutExpired:
                p.kill()
        PROCESOS.pop(nombre, None)
        if not silencioso:
            print(f"🛑 {nombre} detenido.")
    if not silencioso and not PROCESOS:
        print("✅ Nada corriendo. El enlace público quedó muerto.")


def relanzar(proyecto: Path) -> str:
    """Nuevo túnel/servidor (URL nueva) sin recompilar nada."""
    return lanzar(proyecto)


print("✅ Pipeline v2 cargado. Siga con el Paso 1.")

✅ Pipeline v2 cargado. Siga con el Paso 1.


## Paso 1 · Traer el proyecto desde Drive (y sanear lockfiles)

Usted ya descomprimió el ZIP del release y subió la carpeta a
`MyDrive/ProColombia/gestion_conocimiento`. Esta celda la copia al disco local del
runtime (Drive es lento para miles de I/O), localiza la raíz real del proyecto,
verifica su anatomía, **sanea los lockfiles** (hallazgo A07.1) y le dice si llegó el
`dist/` precompilado. ⏱️ ~30–60 s.

In [3]:
PROYECTO = traer_proyecto()

Mounted at /content/drive
✅ Drive montado.
📁 Copiando /content/drive/MyDrive/ProColombia/exportbot → /content/app (sin ['*.egg-info', '.coverage', '.git', '.ipynb_checkpoints']…)
🪪 Integridad del release: 0/0 archivos presentes.
✅ Proyecto listo en /content/app/exportbot · 107 archivos · 54s
🎁 Frontend precompilado: ✅ dist/ presente → se OMITEN Node, npm y build


## Paso 2 · Preparar el entorno (Python — y Node solo si falta `dist/`)

Instala el paquete y el runtime del servidor **en el Python del propio runtime de
Colab** (no crea `venv`: en Colab su arranque de pip falla y deja el entorno sin
`uvicorn`), acelerado con `uv` cuando está disponible. Verifica que `uvicorn`,
`fastapi`, la distribución `exportbot` y los módulos reales del backend queden importables (fail-fast).

Si el `dist/` precompilado llegó en el Paso 1, **aquí no se instala Node ni se
ejecuta npm**: ⏱️ ~30–90 s. Sin `dist/`, compila con latidos y timeouts: ⏱️ ~3–5 min.

In [4]:
PY = preparar_entorno(PROYECTO)

📦 Instalando entorno Python con uv (rápido)…
  $ /usr/local/bin/uv pip install -q --python /usr/bin/python3 -e /content/app/exportbot -r /content/app/exportbot/backend/requirements.txt
✅ Entorno Python listo (/usr/bin/python3) · 6s
♻️ dist/index.html presente: se omiten Node, npm y build (borre dist/ para forzar la recompilación).


## Paso 3 · Lanzar, abrir el túnel y VERIFICAR la URL pública

Levanta Uvicorn desde `backend/` (el backend sirve también el SPA), espera el
endpoint de salud, abre el túnel de Cloudflare y — novedad clave — **consulta la URL
pública desde afuera hasta recibir HTTP 200**. La celda termina con un veredicto
explícito: verificado ✅ o qué log mirar. La URL cambia en cada lanzamiento.

> 🔑 Para consultas con LLM: agregue en los Secrets de Colab al menos una llave
> (`GEMINI_API_KEY`, `GROQ_API_KEY`, …) **antes** de ejecutar esta celda, y active
> el acceso del notebook. Si define `APP_TOKEN`, la API lo exigirá.

In [5]:
URL_PUBLICA = lanzar(PROYECTO)

🔑 Secretos presentes: ['SF_ACCOUNT', 'SF_USER', 'GEMINI_API_KEY', 'GROQ_API_KEY']
✅ Backend arriba en :8000 · {'version': '2.0.0b2'}
⬇️ Descargando cloudflared…
🔎 Verificando la URL pública (/api/salud)…

══════════════════════════════════════════════════════════════════
🌐 URL PÚBLICA:  https://happen-combining-served-improved.trycloudflare.com
══════════════════════════════════════════════════════════════════
✅ VERIFICADO extremo a extremo: HTTP 200 en /api/salud · total 47s
Compártala solo mientras dure la demo. Deténgala con detener().


## Mantener la demo viva · Relanzar · Detener

- **Mantener viva:** deje esta pestaña abierta y el runtime conectado; interactúe con
  el notebook de vez en cuando (`estado()` ayuda). Colab puede cortar sin aviso.
- **Relanzar** (URL nueva, sin recompilar): ejecute la celda de `relanzar(PROYECTO)`.
- **Detener** (mata el enlace público): `detener()`.

In [ ]:
# Detener el servidor y el túnel (cierra el enlace público)
detener()

In [ ]:
# Re-lanzar sin recompilar (genera URL nueva)
URL_PUBLICA = relanzar(PROYECTO)

### Diagnóstico (si algo no arrancó)

Los logs disponibles son `backend`, `tunel`, `npm` y `build`.

In [ ]:
estado()
logs("backend", n=40)   # cambie a logs("tunel"), logs("npm") o logs("build")

## ¿Necesita algo permanente? Las rutas serias

Este notebook es para demos efímeras. Para operación real:

1. **Railway (recomendada, ya preparada):** el proyecto trae `Dockerfile` +
   `railway.toml` verificados. Cree el servicio desde el repositorio y valide
   `GET /api/v1/salud`. Guía y pendientes: `docs/EVIDENCIA_FASE_2A.md`.
2. **Repositorio como única verdad:** publique con el notebook
   `notebooks/Publicar_GitHub.ipynb` siguiendo `docs/MIGRACION_GIT.md` (rama de
   trabajo → PR → CI verde → tag).

### Cómo reutilizar esta plantilla en otro proyecto

Edite **solo la Celda A**: `RUTA_DRIVE`, `BACKEND_DIR`, `APP_ASGI`, `RUTA_SALUD`,
`ENV_APP` y `SECRETS_PASSTHROUGH`. El pipeline (Celda B) no asume nada específico de
este proyecto: localiza cualquier árbol con `pyproject.toml + package.json +
<BACKEND_DIR>/main.py`, sanea lockfiles de forma genérica, aprovecha `dist/`
precompilado si existe y deriva la versión de Node de `package.json → engines`.